# Cleaning Data

In [ ]:
from pathlib import Path
import io
import zipfile
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ZIP_PATH = ROOT / "data" / "Dataset DA Gelar Rasa 2026.zip"
OUTPUT = ROOT / "hasil_cleaning_final"
if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"File belum ada: {ZIP_PATH}")
OUTPUT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    daftar_file = z.namelist()
print("Dataset awal:", ZIP_PATH.name)


In [ ]:
def baca_csv(nama):
    with zipfile.ZipFile(ZIP_PATH) as z:
        return pd.read_csv(io.BytesIO(z.read(nama)), low_memory=False)

def rapikan_teks(df):
    df = df.copy()
    df.columns = df.columns.str.strip()
    for kolom in df.select_dtypes(include=['object', 'string']).columns:
        df[kolom] = df[kolom].astype('string').str.strip()
        df[kolom] = df[kolom].replace('', pd.NA)
    return df

sites = rapikan_teks(baca_csv('sites.csv'))
sensors = rapikan_teks(baca_csv('sensors.csv'))
hospitals = rapikan_teks(baca_csv('hospitals.csv'))
dates_meta = rapikan_teks(baca_csv('metadata/metadata_dates.csv'))
sensor_meta = rapikan_teks(baca_csv('metadata/metadata_sensors.csv'))
health_raw = rapikan_teks(baca_csv('HLT-01_health_outcomes_daily.csv'))
aqms_files = sorted([x for x in daftar_file if x.endswith('_AQMS.csv')])
print('Jumlah file stasiun:', len(aqms_files))


In [ ]:
def audit(df, nama):
    return pd.DataFrame({
        'dataset': nama,
        'kolom': df.columns,
        'tipe_data': df.dtypes.astype(str).values,
        'jumlah_missing': df.isna().sum().values,
        'persen_missing': (df.isna().mean() * 100).round(2).values,
        'jumlah_unik': df.nunique(dropna=True).values
    })

audit_awal = [audit(d, nama) for nama, d in [
    ('sites', sites), ('sensors', sensors), ('hospitals', hospitals),
    ('metadata_dates', dates_meta), ('metadata_sensors', sensor_meta),
    ('health', health_raw)]]
print('Duplikat baris kesehatan:', health_raw.duplicated().sum())
print('Audit awal siap:', len(audit_awal), 'dataset')


## Gabungkan AQMS

In [ ]:
bagian = []
for nama in aqms_files:
    df = rapikan_teks(baca_csv(nama))
    df['file_sumber'] = nama
    df['format_satuan_asli'] = 'ppm' if 'pm25_ppm' in df.columns else 'ug/m3'
    bagian.append(df)
aqms_raw = pd.concat(bagian, ignore_index=True)
audit_awal.append(audit(aqms_raw, 'aqms_gabungan'))
print('Baris AQMS mentah:', len(aqms_raw))
print('Satuan sumber:', aqms_raw['format_satuan_asli'].value_counts().to_dict())
print('Duplikat measurement_id:', aqms_raw['measurement_id'].duplicated().sum())


In [ ]:
master = {}
for nama, df, kunci in [
    ('sites', sites, 'location_id'), ('sensors', sensors, 'sensor_id'),
    ('hospitals', hospitals, 'hospital_id'), ('metadata_dates', dates_meta, 'location_id')]:
    d = df.drop_duplicates().copy()
    print(nama, '| duplikat ID berbeda/berulang:', d[kunci].duplicated().sum())
    master[nama] = d
master['metadata_sensors'] = sensor_meta.drop_duplicates().copy()
master['sensors']['tanggal_instalasi'] = pd.to_datetime(master['sensors']['tanggal_instalasi'], errors='coerce')
for k in ['date_start','date_end']:
    master['metadata_dates'][k] = pd.to_datetime(master['metadata_dates'][k], errors='coerce')
for k in ['latitude_dd','longitude_dd','distance_to_road_m','traffic_intensity_per_h']:
    master['sites'][k] = pd.to_numeric(master['sites'][k], errors='coerce')
for k in ['latitude','longitude']:
    master['sensors'][k] = pd.to_numeric(master['sensors'][k], errors='coerce')
print('ID lokasi tidak cocok:', (~master['sensors']['location_id'].isin(master['sites']['location_id'])).sum())


In [ ]:
aqms = aqms_raw.drop_duplicates().copy()
for k in ['timestamp_start', 'timestamp_end']:
    aqms[k] = pd.to_datetime(aqms[k], errors='coerce', format='mixed')
polutan = ['pm25', 'pm10', 'so2', 'co', 'o3', 'no2']
meteo = ['temperature_c', 'humidity_pct', 'wind_speed_ms', 'wind_direction_deg']
for p in polutan:
    for suffix in ['ug_m3', 'ppm']:
        k = f'{p}_{suffix}'
        if k in aqms.columns:
            aqms[k] = pd.to_numeric(aqms[k], errors='coerce')
for k in meteo:
    aqms[k] = pd.to_numeric(aqms[k], errors='coerce')

qc = aqms['qc_flag'].astype('string').str.upper().str.strip()
print('QC sebelum cleaning:', qc.value_counts(dropna=False).to_dict())
aqms = aqms.loc[qc.isin(['VALID', 'SUSPECT'])].copy()
qc = qc.loc[aqms.index]
kolom_ukur = [f'{p}_{s}' for p in polutan for s in ['ug_m3','ppm'] if f'{p}_{s}' in aqms.columns] + meteo
aqms.loc[qc.eq('SUSPECT'), kolom_ukur] = np.nan
aqms = aqms.drop(columns=['qc_flag'])

for k in kolom_ukur:
    if k not in ['temperature_c', 'wind_direction_deg']:
        aqms.loc[aqms[k] < 0, k] = np.nan
aqms.loc[~aqms['humidity_pct'].between(0,100), 'humidity_pct'] = np.nan
aqms.loc[~aqms['wind_direction_deg'].between(0,360), 'wind_direction_deg'] = np.nan
durasi = (aqms['timestamp_end'] - aqms['timestamp_start']).dt.total_seconds()/60
aqms = aqms.loc[aqms['timestamp_start'].notna() & (durasi > 0) & aqms['timestamp_start'].dt.year.between(2024,2026)].copy()
aqms['tanggal'] = aqms['timestamp_start'].dt.normalize()
aqms['jam'] = aqms['timestamp_start'].dt.hour
print('Baris AQMS sesudah QC dan validasi waktu:', len(aqms))


## Standarkan satuan

In [ ]:
berat_molekul = {'so2':64.066, 'co':28.01, 'o3':48.00, 'no2':46.0055}
for p in polutan:
    kolom = f'{p}_ug_m3'
    aqms[kolom] = aqms[kolom].astype(float)
    if p in berat_molekul:
        # Konversi gas: ppm × massa molekul × 1000 / 24,45 (25 °C, 1 atm)
        hasil_konversi = aqms[f'{p}_ppm'] * berat_molekul[p] * 1000 / 24.45
        aqms[kolom] = aqms[kolom].fillna(hasil_konversi)
print('Konversi gas selesai; PM dengan label ppm tetap kosong pada kolom µg/m³.')


In [ ]:
sensor_lookup = master['sensors'][['sensor_id','location_id']]
site_lookup = master['sites'][['location_id','provinsi','kota_kabupaten','kecamatan','region_group']]
aqms = aqms.merge(sensor_lookup, on='sensor_id', how='left', validate='many_to_one', indicator='status_sensor')
aqms = aqms.merge(site_lookup, on='location_id', how='left', validate='many_to_one', indicator='status_lokasi')
print(aqms['status_sensor'].value_counts().to_dict())
print(aqms['status_lokasi'].value_counts().to_dict())


## Pisahkan data AQMS dan cuaca

In [ ]:
kunci = ['measurement_id','sensor_id','location_id','timestamp_start','timestamp_end',
         'tanggal','jam','provinsi','kota_kabupaten','kecamatan','region_group']
aqms_clean = aqms[kunci + [f'{p}_ug_m3' for p in polutan]].copy()
meteo_clean = aqms[kunci + meteo].copy()
print('AQMS:', aqms_clean.shape, '| Meteorologi:', meteo_clean.shape)


## Bersihkan data kesehatan

In [ ]:
health = health_raw.drop_duplicates().copy()
health['date'] = pd.to_datetime(health['date'], errors='coerce').dt.normalize()
health['hospital_id'] = health['hospital_id'].astype('string').str.strip()
kolom_kasus = ['resp_admissions_under5','resp_admissions_over60','asthma_emergency_cases',
               'copd_admissions','resp','cvd','death']
for k in kolom_kasus:
    health[k] = pd.to_numeric(health[k], errors='coerce')
    health.loc[health[k] < 0, k] = np.nan
for k in ['temp_c','dptp_c','rhum_pct','time']:
    health[k] = pd.to_numeric(health[k], errors='coerce')
health.loc[~health['rhum_pct'].between(0,100),'rhum_pct'] = np.nan
print('Hospital ID tidak terdaftar:', (~health['hospital_id'].isin(master['hospitals']['hospital_id'])).sum())
print('Duplikat hospital-tanggal:', health.duplicated(['hospital_id','date']).sum())
print('Missing kasus:', health[kolom_kasus].isna().sum().to_dict())


## Gabungkan data rumah sakit

In [ ]:
health_clean = health.merge(master['hospitals'], on='hospital_id', how='left', validate='many_to_one', indicator='status_hospital')
print(health_clean['status_hospital'].value_counts().to_dict())
print('Rentang tanggal kesehatan:', health_clean['date'].min(), health_clean['date'].max())
print('Rentang tanggal AQMS:', aqms_clean['tanggal'].min(), aqms_clean['tanggal'].max())
health_clean = health_clean.drop(columns=['_merge', 'hospital_terdaftar'], errors='ignore')


## Gabungkan metadata stasiun

In [ ]:
stations_clean = master['sensors'].merge(master['sites'], on='location_id', how='left', validate='many_to_one', suffixes=('_sensor','_site'))
stations_clean = stations_clean.merge(master['metadata_dates'], on='location_id', how='left', validate='many_to_one', suffixes=('','_periode'))
print('Metadata stasiun:', stations_clean.shape)
print('Koordinat kosong:', stations_clean[['latitude','longitude']].isna().sum().to_dict())


In [ ]:
kelengkapan_stasiun = (
    aqms_clean.groupby("sensor_id")
    .agg(
        jumlah_pengukuran=("measurement_id", "size"),
        jumlah_pm25_terisi=("pm25_ug_m3", "count")
    )
    .reset_index()
)

kelengkapan_stasiun["persen_pm25"] = (
    kelengkapan_stasiun["jumlah_pm25_terisi"]
    / kelengkapan_stasiun["jumlah_pengukuran"]
    * 100
).round(2)

print(kelengkapan_stasiun.to_string(index=False))


## Kamus data

In [ ]:
deskripsi = {
    'measurement_id':('ID unik pengukuran','-'), 'sensor_id':('ID sensor','-'),
    'location_id':('ID lokasi stasiun','-'), 'hospital_id':('ID rumah sakit','-'),
    'timestamp_start':('Waktu awal pengukuran','waktu lokal sesuai sumber'),
    'timestamp_end':('Waktu akhir pengukuran','waktu lokal sesuai sumber'),
    'tanggal':('Tanggal awal pengukuran','tanggal'), 'date':('Tanggal pencatatan kesehatan','tanggal'),
    'qc_flag':('Penanda kualitas dari data sumber','-'),
    'layak_analisis':('QC valid, timestamp dan lokasi tersedia, durasi positif','boolean'),
    'temperature_c':('Suhu udara','°C'), 'humidity_pct':('Kelembapan relatif','%'),
    'wind_speed_ms':('Kecepatan angin','m/s'), 'wind_direction_deg':('Arah angin','derajat'),
    'resp_admissions_under5':('Rawat inap pernapasan balita','kasus'),
    'resp_admissions_over60':('Rawat inap pernapasan lansia','kasus'),
    'asthma_emergency_cases':('Kasus gawat darurat asma','kasus'),
    'copd_admissions':('Rawat inap COPD','kasus')}
kamus=[]
for nama, df in [('aqms',aqms_clean),('meteorologi',meteo_clean),('kesehatan',health_clean),('stasiun',stations_clean)]:
    for k in df.columns:
        definisi,satuan = deskripsi.get(k,('Periksa definisi pada metadata/sumber','-'))
        if k.endswith('_ug_m3'): definisi,satuan = 'Konsentrasi '+k.split('_')[0]+' dalam satuan standar','µg/m³'
        if k.endswith('_ppm'): definisi,satuan = 'Nilai ppm asli (jika tersedia)','ppm (sumber)'
        kamus.append({'dataset':nama,'kolom':k,'tipe_data':str(df[k].dtype),'definisi':definisi,'satuan':satuan})
data_dictionary = pd.DataFrame(kamus)
print('Baris kamus:',len(data_dictionary))


In [ ]:
keputusan = pd.DataFrame([
    ['Duplikat','Hapus hanya duplikat baris identik','ID ganda dengan isi berbeda harus direview'],
    ['Missing','Pertahankan sebagai NaN','Tidak mengisi rata-rata otomatis'],
    ['QC','Simpan pengukuran VALID; kosongkan ukuran SUSPECT; keluarkan status QC lain','QC asli tetap dalam ZIP sumber'],
    ['Nilai negatif','Ubah nilai negatif konsentrasi/kasus menjadi NaN','Nilai asli masih tersedia dalam ZIP'],
    ['Outlier','Tidak dihapus otomatis','Evaluasi konteks dan QC sebelum keputusan'],
    ['Gas ppm','Konversi asumsi 25°C, 1 atm: ppm × MW × 1000 / 24.45','Asumsi perlu dikonfirmasi jika kondisi aktual diperlukan'],
    ['PM ppm','Tidak dikonversi otomatis','PM2.5/PM10 bukan gas; konfirmasi label/metadata'],
    ['Tanggal','Parse mixed timestamp; tandai tahun di luar 2024–2026; tanggal AQMS dari timestamp_start','Periksa zona waktu dan pergeseran menit pada 3 stasiun'],
    ['Integrasi','Join AQMS-sensor-lokasi dan kesehatan-rumah sakit','Belum menggabungkan AQMS dan kesehatan secara paksa'],
    ['Coverage','Rasio PM2.5 terisi dibanding baris pengukuran per stasiun','Bukan uptime nominal stasiun']
],columns=['aspek','tindakan','catatan'])
print(keputusan.to_string(index=False))


In [ ]:
audit_awal_df = pd.concat(audit_awal, ignore_index=True)
audit_akhir = pd.concat([
    audit(aqms_clean, "aqms_final"),
    audit(meteo_clean, "meteorologi_final"),
    audit(health_clean, "kesehatan_final"),
    audit(stations_clean, "stasiun_final"),
], ignore_index=True)
ringkasan = pd.DataFrame([
    {"dataset": nama, "jumlah_baris": len(df), "jumlah_kolom": len(df.columns)}
    for nama, df in [
        ("aqms", aqms_clean), ("meteorologi", meteo_clean),
        ("kesehatan", health_clean), ("stasiun", stations_clean)
    ]
])


In [ ]:
files_out = {
    '01_aqms_clean.csv':aqms_clean,
    '02_meteorologi_clean.csv':meteo_clean,
    '03_kesehatan_clean.csv':health_clean,
    '04_metadata_stasiun_clean.csv':stations_clean,
    '05_data_dictionary.csv':data_dictionary,
    '06_audit_awal.csv':audit_awal_df,
    '07_audit_akhir.csv':audit_akhir,
    '08_ringkasan_baris.csv':ringkasan,
    '09_kelengkapan_per_stasiun.csv':kelengkapan_stasiun,
    '10_log_keputusan.csv':keputusan
}
for nama,df in files_out.items():
    df.to_csv(OUTPUT/nama,index=False,encoding='utf-8-sig')
    print('Disimpan:',nama,df.shape)
print('Selesai. Folder:', OUTPUT.resolve())


In [ ]:
print('Jumlah baris AQMS final:', len(aqms_clean))
print('Jumlah baris meteorologi final:', len(meteo_clean))
print('Kolom AQMS final:', aqms_clean.columns.tolist())
print('Selesai. Hasil cleaning:', OUTPUT.resolve())
